### Imports

In [7]:
import os, warnings
import faiss
from llama_index.core import (
    SimpleDirectoryReader, VectorStoreIndex, StorageContext,
    Settings, PromptTemplate, load_index_from_storage,
)
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.vector_stores.faiss import FaissVectorStore

from ragas import evaluate, EvaluationDataset, RunConfig
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
with warnings.catch_warnings():
    warnings.simplefilter("ignore", DeprecationWarning)   
    from ragas.metrics import (
        faithfulness, answer_relevancy, context_precision,
        context_recall, answer_similarity,
    )
from langchain_ollama import ChatOllama, OllamaEmbeddings 

### RAG Pipeline

In [ ]:
model = Ollama(model = "llama3.2:3b", temperature = 0.0, request_timeout = 180.0)
embedding = OllamaEmbedding(model_name="nomic-embed-text")
Settings.llm = model
Settings.embed_model = embedding

# Setting up non-volatile Faiss DB
faiss_dir = "./faiss_index"
os.makedirs(faiss_dir, exist_ok=True)

# Reloading the index instead of re-embedding
if os.path.exists(os.path.join(faiss_dir, "docstore.json")):
    vector_store = FaissVectorStore.from_persist_dir(faiss_dir)
    storage_context = StorageContext.from_defaults(vector_store=vector_store, persist_dir=faiss_dir)
    index = load_index_from_storage(storage_context)
else:
    documents = SimpleDirectoryReader(input_dir="./data_sample").load_data()
    nodes = SentenceSplitter(chunk_size=512, chunk_overlap=50).get_nodes_from_documents(documents)

    dim = len(embedding.get_text_embedding("test"))          
    vector_store = FaissVectorStore(faiss_index=faiss.IndexFlatL2(dim))   
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    index = VectorStoreIndex(nodes, storage_context=storage_context)
    index.storage_context.persist(persist_dir=faiss_dir)

qa_prompt = PromptTemplate(
    "Context information:\n{context_str}\n\n"
    "Answer the question using ONLY the context above. Be complete, and copy numbers, "
    "ranges and units exactly as written. If several chemistries/items are in the context, "
    "compare them. If the answer is not in the context, say \"I don't know\".\n"
    "Question: {query_str}\nAnswer: "
)      

query_engine = index.as_query_engine(similarity_top_k=5, text_qa_template=qa_prompt)

#### Questions, Contexts & Ground truth

In [11]:
questions = [
    "What is the optimal tilt angle for a site at 40 degrees north latitude?",
    "How often should solar panels be cleaned?",
    "What is the cycle life of LiFePO4 cells at 80% depth of discharge?",
    "What are the benefits of using LiFePO4 batteries in residential storage systems?",
    "What temperature range can lithium iron phosphate batteries operate in?",
    "What type of battery has the lowest initial cost but highest lifetime cost?",
    "What is the energy density of NMC batteries compared to other chemistries?",
    "How does shading affect solar panel output?",
    "Why are BMS components critical for lithium battery systems?",
    "What is the recommended roof load capacity for residential solar installations?"
]
ground_truths = [
    "The optimal tilt angle equals the latitude of the installation site. For a 40 degrees north latitude, the panel should be tilted at 40 degrees from horizontal.",
    "Solar panels should be cleaned every 2-3 months or after heavy dust storms or bird droppings to maintain efficiency and performance.",
    "LiFePO4 cells typically have a cycle life of 3,000-6,000 cycles at 80% depth of discharge.",
    "LiFePO4 batteries offer excellent thermal stability (non-flammable electrolyte), long cycle life, and high safety for residential storage applications.",
    "Lithium iron phosphate batteries can operate in temperatures from 0 to 45 degrees Celsius.",
    "Lead-acid (AGM/Gel) batteries have the lowest initial cost but higher lifetime costs due to frequent replacement requirements.",
    "NMC batteries have an energy density of 150-220 Wh/kg, which is higher than lead-acid (30-40 Wh/kg) but lower than LiFePO4 (90-160 Wh/kg).",
    "Even partial shading on a single cell can reduce the output of an entire string by up to 80%.",
    "The Battery Management System (BMS) is critical for lithium battery systems as it monitors and controls key parameters including cell voltage, temperature, current, and state of charge for safety and performance.",
    "Residential roofs must support at least 30 kg per square meter dead load plus 1.5 kN per square meter live load in snow regions."
]
contexts = [
    "For fixed installations, the optimal tilt angle equals the latitude. At 40 degrees north latitude, this means a 40-degree tilt is optimal for solar panels.",
    "Solar panel maintenance includes cleaning every 2-3 months or after heavy dust storms or bird droppings to maintain efficiency and performance.",
    "Lithium Iron Phosphate (LiFePO4): Cycle life: 3,000-6,000 cycles at 80% depth of discharge",
    "Lithium Iron Phosphate (LiFePO4): Thermal stability: Excellent, non-flammable electrolyte. Best for: Long-duration residential storage, high safety.",
    "Lithium Iron Phosphate (LiFePO4): Operating temperature: 0 to 45 degrees C",
    "Lead-Acid (AGM / Gel): Cost: Lowest initial, highest lifetime (due to replacement)",
    "Nickel Manganese Cobalt (NMC): Energy density: 150-220 Wh/kg",
    "Even partial shading on a single cell can reduce the output of an entire string by up to 80%. A shading study using a solace sensor or PV*SOL software must be performed.",
    "The BMS is the critical safety and performance component of any lithium battery system. It monitors and controls: Cell voltage, Temperature, Current, State of Charge (SOC) estimation",
    "Residential roofs must support a dead load of at least 30 kg per square meter for the mounting hardware and panels, plus a live load of 1.5 kN per square meter for snow regions."
]

### Result

In [12]:
results = []

for i, question in enumerate(questions):
    print("\n" + "=" * 60)
    print("Q:", question)
    r = query_engine.query(question)
    print("A:", r.response)
    results.append({
        "question": question,
        "answer": str(r.response),
        "ground_truth": ground_truths[i],
        "contexts": [node.get_content() for node in r.source_nodes]
    })


Q: What is the optimal tilt angle for a site at 40 degrees north latitude?
A: The optimal tilt angle for a site at 40 degrees north latitude is 40 degrees from horizontal.

Q: How often should solar panels be cleaned?
A: Quarterly, with a soft brush and deionised water, never using abrasive cleaners or pressure washers above 40 bar, as these can crack the tempered glass cover.

Q: What is the cycle life of LiFePO4 cells at 80% depth of discharge?
A: 3,000-6,000 cycles.

Q: What are the benefits of using LiFePO4 batteries in residential storage systems?
A: According to the context, the benefits of using LiFePO4 batteries in residential storage systems are:

* High energy density: 90-160 Wh/kg
* Excellent thermal stability: non-flammable electrolyte
* Long cycle life: 3,000-6,000 cycles at 80% depth of discharge
* High safety: moderate cost
* Best for: Long-duration residential storage, high safety

Note that the cost is described as "moderate", which may seem counterintuitive given the

### Evaluation